In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Customer data") \
    .getOrCreate()

df=spark.read.csv("customers_data.csv",header=True,inferSchema=True)
df.show()

+-----------+-----+----+-----------+-------------+-------+--------------------+--------------+----------+
|customer_id| name| age|     salary|      country|   city|               email|         phone| join_date|
+-----------+-----+----+-----------+-------------+-------+--------------------+--------------+----------+
|          1| John| 150|    -1000.0|        U.S.A|  Delhi|floresrandall@liv...|    9876543210|12-05-2024|
|          1| John|  25|     5000.0|        U.S.A|  Delhi|                NULL|    9876543210|2024/01/01|
|          2| mike|NULL|       NULL|United States|Chennai| sarah97@hotmail.com|          NULL|31-13-2024|
|          4| mike|  30|    -1000.0|          USA|Chennai|       invalid_email|    9876543210|31-13-2024|
|          5|david|  45|     5000.0|        india|   NULL|                NULL|    9876543210|      NULL|
|          6|david|  45|       NULL|United States|   NULL|michael17@mcbride...|    9876543210|12-05-2024|
|          7|SARAH|  25|    10000.0|          

In [2]:
from pyspark.sql.functions import col,sum

df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df.columns
]).show()

+-----------+----+---+------+-------+----+-----+-----+---------+
|customer_id|name|age|salary|country|city|email|phone|join_date|
+-----------+----+---+------+-------+----+-----+-----+---------+
|          0| 159|124|   174|      0| 345|  260|  214|      199|
+-----------+----+---+------+-------+----+-----+-----+---------+



In [3]:
df_drop=df.na.drop(subset=["customer_id","salary"])
df_drop.show()

+-----------+-----+----+-----------+-------------+-------+--------------------+--------------+----------+
|customer_id| name| age|     salary|      country|   city|               email|         phone| join_date|
+-----------+-----+----+-----------+-------------+-------+--------------------+--------------+----------+
|          1| John| 150|    -1000.0|        U.S.A|  Delhi|floresrandall@liv...|    9876543210|12-05-2024|
|          1| John|  25|     5000.0|        U.S.A|  Delhi|                NULL|    9876543210|2024/01/01|
|          4| mike|  30|    -1000.0|          USA|Chennai|       invalid_email|    9876543210|31-13-2024|
|          5|david|  45|     5000.0|        india|   NULL|                NULL|    9876543210|      NULL|
|          7|SARAH|  25|    10000.0|          USA|   NULL|       invalid_email|          NULL|31-13-2024|
|          8|SARAH|  45|    10000.0|        india| Mumbai|                NULL|    9876543210|2024/01/01|
|          9| mike|  12|9.9999999E7|United Sta

In [4]:
df_fill=df.na.fill({
    "name":"Unknown",
    "city":"Unknown",
    "salary":0
})

df_fill.show()

+-----------+-------+----+-----------+-------------+-------+--------------------+--------------+----------+
|customer_id|   name| age|     salary|      country|   city|               email|         phone| join_date|
+-----------+-------+----+-----------+-------------+-------+--------------------+--------------+----------+
|          1|   John| 150|    -1000.0|        U.S.A|  Delhi|floresrandall@liv...|    9876543210|12-05-2024|
|          1|   John|  25|     5000.0|        U.S.A|  Delhi|                NULL|    9876543210|2024/01/01|
|          2|   mike|NULL|        0.0|United States|Chennai| sarah97@hotmail.com|          NULL|31-13-2024|
|          4|   mike|  30|    -1000.0|          USA|Chennai|       invalid_email|    9876543210|31-13-2024|
|          5|  david|  45|     5000.0|        india|Unknown|                NULL|    9876543210|      NULL|
|          6|  david|  45|        0.0|United States|Unknown|michael17@mcbride...|    9876543210|12-05-2024|
|          7|  SARAH|  25|  

In [5]:
from pyspark.sql.functions import mean
mean_sal=df.select(mean("salary")).collect()[0][0]
print(mean_sal)

df_fill_mean=df.na.fill({"salary":mean_sal})
df_fill_mean.show()

23326416.69968051
+-----------+-----+----+-------------------+-------------+-------+--------------------+--------------+----------+
|customer_id| name| age|             salary|      country|   city|               email|         phone| join_date|
+-----------+-----+----+-------------------+-------------+-------+--------------------+--------------+----------+
|          1| John| 150|            -1000.0|        U.S.A|  Delhi|floresrandall@liv...|    9876543210|12-05-2024|
|          1| John|  25|             5000.0|        U.S.A|  Delhi|                NULL|    9876543210|2024/01/01|
|          2| mike|NULL|2.332641669968051E7|United States|Chennai| sarah97@hotmail.com|          NULL|31-13-2024|
|          4| mike|  30|            -1000.0|          USA|Chennai|       invalid_email|    9876543210|31-13-2024|
|          5|david|  45|             5000.0|        india|   NULL|                NULL|    9876543210|      NULL|
|          6|david|  45|2.332641669968051E7|United States|   NULL|mich

In [6]:
from pyspark.sql.functions import mode
mode_city=df.select(mode("city")).collect()[0][0]
print(mode_city)
df_fill_mode=df.na.fill({"city":mode_city})
df_fill_mode.show()

Mumbai
+-----------+-----+----+-----------+-------------+-------+--------------------+--------------+----------+
|customer_id| name| age|     salary|      country|   city|               email|         phone| join_date|
+-----------+-----+----+-----------+-------------+-------+--------------------+--------------+----------+
|          1| John| 150|    -1000.0|        U.S.A|  Delhi|floresrandall@liv...|    9876543210|12-05-2024|
|          1| John|  25|     5000.0|        U.S.A|  Delhi|                NULL|    9876543210|2024/01/01|
|          2| mike|NULL|       NULL|United States|Chennai| sarah97@hotmail.com|          NULL|31-13-2024|
|          4| mike|  30|    -1000.0|          USA|Chennai|       invalid_email|    9876543210|31-13-2024|
|          5|david|  45|     5000.0|        india| Mumbai|                NULL|    9876543210|      NULL|
|          6|david|  45|       NULL|United States| Mumbai|michael17@mcbride...|    9876543210|12-05-2024|
|          7|SARAH|  25|    10000.0|   

In [7]:
df.dropDuplicates().count()

800

In [8]:
from pyspark.sql.functions import trim,lower,upper,initcap,col
df=df.withColumn(
    "name",
    trim(col("name"))
)

df=df.withColumn(
    "name",
    initcap(col("name"))
)

df.show()

+-----------+-----+----+-----------+-------------+-------+--------------------+--------------+----------+
|customer_id| name| age|     salary|      country|   city|               email|         phone| join_date|
+-----------+-----+----+-----------+-------------+-------+--------------------+--------------+----------+
|          1| John| 150|    -1000.0|        U.S.A|  Delhi|floresrandall@liv...|    9876543210|12-05-2024|
|          1| John|  25|     5000.0|        U.S.A|  Delhi|                NULL|    9876543210|2024/01/01|
|          2| Mike|NULL|       NULL|United States|Chennai| sarah97@hotmail.com|          NULL|31-13-2024|
|          4| Mike|  30|    -1000.0|          USA|Chennai|       invalid_email|    9876543210|31-13-2024|
|          5|David|  45|     5000.0|        india|   NULL|                NULL|    9876543210|      NULL|
|          6|David|  45|       NULL|United States|   NULL|michael17@mcbride...|    9876543210|12-05-2024|
|          7|Sarah|  25|    10000.0|          

In [9]:
from pyspark.sql.functions import regexp_replace

df=df.withColumn(
    "phone",
    regexp_replace("phone","[^0-9]","")
)

df.show()

+-----------+-----+----+-----------+-------------+-------+--------------------+----------+----------+
|customer_id| name| age|     salary|      country|   city|               email|     phone| join_date|
+-----------+-----+----+-----------+-------------+-------+--------------------+----------+----------+
|          1| John| 150|    -1000.0|        U.S.A|  Delhi|floresrandall@liv...|9876543210|12-05-2024|
|          1| John|  25|     5000.0|        U.S.A|  Delhi|                NULL|9876543210|2024/01/01|
|          2| Mike|NULL|       NULL|United States|Chennai| sarah97@hotmail.com|      NULL|31-13-2024|
|          4| Mike|  30|    -1000.0|          USA|Chennai|       invalid_email|9876543210|31-13-2024|
|          5|David|  45|     5000.0|        india|   NULL|                NULL|9876543210|      NULL|
|          6|David|  45|       NULL|United States|   NULL|michael17@mcbride...|9876543210|12-05-2024|
|          7|Sarah|  25|    10000.0|          USA|   NULL|       invalid_email|   

In [10]:
df_valid_phone=df.filter(
    col("phone").rlike("^[0-9]{10}$")
)

df_valid_phone.show()

+-----------+-----+----+-----------+-------------+-------+--------------------+----------+----------+
|customer_id| name| age|     salary|      country|   city|               email|     phone| join_date|
+-----------+-----+----+-----------+-------------+-------+--------------------+----------+----------+
|          1| John| 150|    -1000.0|        U.S.A|  Delhi|floresrandall@liv...|9876543210|12-05-2024|
|          1| John|  25|     5000.0|        U.S.A|  Delhi|                NULL|9876543210|2024/01/01|
|          4| Mike|  30|    -1000.0|          USA|Chennai|       invalid_email|9876543210|31-13-2024|
|          5|David|  45|     5000.0|        india|   NULL|                NULL|9876543210|      NULL|
|          6|David|  45|       NULL|United States|   NULL|michael17@mcbride...|9876543210|12-05-2024|
|          8|Sarah|  45|    10000.0|        india| Mumbai|                NULL|9876543210|2024/01/01|
|         10| NULL| 150|9.9999999E7|        U.S.A| Mumbai|leroygarcia@yahoo...|987

In [11]:
df_valid_email=df_valid_phone.filter(
    col("email").rlike(
        r'^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$'
    )
)

df_valid_email.show()

+-----------+-----+----+-----------+-------------+-------+--------------------+----------+----------+
|customer_id| name| age|     salary|      country|   city|               email|     phone| join_date|
+-----------+-----+----+-----------+-------------+-------+--------------------+----------+----------+
|          1| John| 150|    -1000.0|        U.S.A|  Delhi|floresrandall@liv...|9876543210|12-05-2024|
|          6|David|  45|       NULL|United States|   NULL|michael17@mcbride...|9876543210|12-05-2024|
|         10| NULL| 150|9.9999999E7|        U.S.A| Mumbai|leroygarcia@yahoo...|9876543210|31-13-2024|
|         13|David|NULL|       NULL|United States|   NULL|   scott90@yahoo.com|9876543210|31-13-2024|
|         18| Mike| abc|     5000.0|          USA|   NULL|xfischer@johnson.org|9876543210|2024/01/01|
|         21| John|  45|     5000.0|        INDIA|   NULL|browndonna@hotmai...|9876543210|12-05-2024|
|         22| John|  30|     5000.0|          USA| Mumbai|   david23@yahoo.com|987

In [16]:
from pyspark.sql.functions import expr
df=df.withColumn(
    "age_clean",
    expr("try_cast(age as int)")
)

df.show()

+-----------+-----+----+-------+-------------+-------+--------------------+----------+----------+---------+
|customer_id| name| age| salary|      country|   city|               email|     phone| join_date|age_clean|
+-----------+-----+----+-------+-------------+-------+--------------------+----------+----------+---------+
|          1| John| 150|    0.0|        U.S.A|  Delhi|floresrandall@liv...|9876543210|12-05-2024|      150|
|          1| John|  25| 5000.0|        U.S.A|  Delhi|                NULL|9876543210|2024/01/01|       25|
|          2| Mike|NULL|   NULL|United States|Chennai| sarah97@hotmail.com|      NULL|31-13-2024|     NULL|
|          4| Mike|  30|    0.0|          USA|Chennai|       invalid_email|9876543210|31-13-2024|       30|
|          5|David|  45| 5000.0|        india|   NULL|                NULL|9876543210|      NULL|       45|
|          6|David|  45|   NULL|United States|   NULL|michael17@mcbride...|9876543210|12-05-2024|       45|
|          7|Sarah|  25|1000

In [13]:
from pyspark.sql.functions import when
q1,q2,q3=df.approxQuantile(
    "age_clean",[0.25,0.5,0.75],0
)
iqr=q3-q1

upper_limit=q3+1.5*iqr
lower_limit=q1-1.5*iqr

df=df.withColumn(
    "age_clean",
    when(col("age_clean")>upper_limit,q2).
    when(col("age_clean")<lower_limit,q2).
    otherwise(col("age_clean"))
)
df.show()

+-----------+-----+----+-----------+-------------+-------+--------------------+----------+----------+---------+
|customer_id| name| age|     salary|      country|   city|               email|     phone| join_date|age_clean|
+-----------+-----+----+-----------+-------------+-------+--------------------+----------+----------+---------+
|          1| John| 150|    -1000.0|        U.S.A|  Delhi|floresrandall@liv...|9876543210|12-05-2024|     45.0|
|          1| John|  25|     5000.0|        U.S.A|  Delhi|                NULL|9876543210|2024/01/01|     25.0|
|          2| Mike|NULL|       NULL|United States|Chennai| sarah97@hotmail.com|      NULL|31-13-2024|     NULL|
|          4| Mike|  30|    -1000.0|          USA|Chennai|       invalid_email|9876543210|31-13-2024|     30.0|
|          5|David|  45|     5000.0|        india|   NULL|                NULL|9876543210|      NULL|     45.0|
|          6|David|  45|       NULL|United States|   NULL|michael17@mcbride...|9876543210|12-05-2024|   

In [14]:
print(q1,q2,q3)

25.0 45.0 45.0


In [15]:
from pyspark.sql.functions import min, max, when, col
df = df.withColumn(
    "salary",
    when(col("salary") < 0, 0)
    .otherwise(col("salary"))
)
sq1, sq2, sq3 = df.approxQuantile(
    "salary",
    [0.25, 0.5, 0.75],
    0
)

siqr = sq3 - sq1

minimum = df.agg(
    min("salary")
).collect()[0][0]

print("min : ", minimum)

maximum = df.agg(
    max("salary")
).collect()[0][0]

print("max : ", maximum)

s_upper_limit = sq3 + 1.5 * siqr
s_lower_limit = sq1 - 1.5 * siqr

print("Q1:", sq1)
print("Median:", sq2)
print("Q3:", sq3)

df = df.withColumn(
    "salary",
    when(col("salary") > s_upper_limit, sq2)
    .when(col("salary") < s_lower_limit, sq2)
    .otherwise(col("salary"))
)

df.show()

min :  0.0
max :  99999999.0
Q1: 5000.0
Median: 5000.0
Q3: 10000.0
+-----------+-----+----+-------+-------------+-------+--------------------+----------+----------+---------+
|customer_id| name| age| salary|      country|   city|               email|     phone| join_date|age_clean|
+-----------+-----+----+-------+-------------+-------+--------------------+----------+----------+---------+
|          1| John| 150|    0.0|        U.S.A|  Delhi|floresrandall@liv...|9876543210|12-05-2024|     45.0|
|          1| John|  25| 5000.0|        U.S.A|  Delhi|                NULL|9876543210|2024/01/01|     25.0|
|          2| Mike|NULL|   NULL|United States|Chennai| sarah97@hotmail.com|      NULL|31-13-2024|     NULL|
|          4| Mike|  30|    0.0|          USA|Chennai|       invalid_email|9876543210|31-13-2024|     30.0|
|          5|David|  45| 5000.0|        india|   NULL|                NULL|9876543210|      NULL|     45.0|
|          6|David|  45|   NULL|United States|   NULL|michael17@mcbri

In [19]:
df=df.replace(
    ["U.S.A","United States"],
    "USA",
    "country"
)
df=df.replace(
    ["INDIA","india"],
    "India",
    "country"
)

df.show()

+-----------+-----+----+-------+-------+-------+--------------------+----------+----------+---------+
|customer_id| name| age| salary|country|   city|               email|     phone| join_date|age_clean|
+-----------+-----+----+-------+-------+-------+--------------------+----------+----------+---------+
|          1| John| 150|    0.0|    USA|  Delhi|floresrandall@liv...|9876543210|12-05-2024|      150|
|          1| John|  25| 5000.0|    USA|  Delhi|                NULL|9876543210|2024/01/01|       25|
|          2| Mike|NULL|   NULL|    USA|Chennai| sarah97@hotmail.com|      NULL|31-13-2024|     NULL|
|          4| Mike|  30|    0.0|    USA|Chennai|       invalid_email|9876543210|31-13-2024|       30|
|          5|David|  45| 5000.0|  India|   NULL|                NULL|9876543210|      NULL|       45|
|          6|David|  45|   NULL|    USA|   NULL|michael17@mcbride...|9876543210|12-05-2024|       45|
|          7|Sarah|  25|10000.0|    USA|   NULL|       invalid_email|      NULL|31

In [20]:
df=df.withColumn(
    "city",
    when(col("city")=="",None).
    otherwise(col("city"))
)
df.show()

+-----------+-----+----+-------+-------+-------+--------------------+----------+----------+---------+
|customer_id| name| age| salary|country|   city|               email|     phone| join_date|age_clean|
+-----------+-----+----+-------+-------+-------+--------------------+----------+----------+---------+
|          1| John| 150|    0.0|    USA|  Delhi|floresrandall@liv...|9876543210|12-05-2024|      150|
|          1| John|  25| 5000.0|    USA|  Delhi|                NULL|9876543210|2024/01/01|       25|
|          2| Mike|NULL|   NULL|    USA|Chennai| sarah97@hotmail.com|      NULL|31-13-2024|     NULL|
|          4| Mike|  30|    0.0|    USA|Chennai|       invalid_email|9876543210|31-13-2024|       30|
|          5|David|  45| 5000.0|  India|   NULL|                NULL|9876543210|      NULL|       45|
|          6|David|  45|   NULL|    USA|   NULL|michael17@mcbride...|9876543210|12-05-2024|       45|
|          7|Sarah|  25|10000.0|    USA|   NULL|       invalid_email|      NULL|31

In [ ]:
df=df.withColumn(
    "date_valid_flag",
    when(col("join_date_clean").isNotNull(),"VALID").otherwise("INVALID")
)

3.5.1
